In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision  # Because it provides ready-made utilities for: Image datasets  Image transformations  Pretrained models Image loading tools
from torchvision.datasets import CIFAR10

In [2]:
# DAtasets and Dataloaders
from torch.utils.data import Dataset , DataLoader
import torchvision.transforms as transforms  # use for multiple transfrmation of image
transform = transforms.Compose([       # use for chaining multiple transformation
    transforms.ToTensor(),             # convert to tensor and scale it on (0,1)
    transforms.Normalize( (0.5,0.5,0.5), (0.5,0.5,0.5))     # to range it (-1, +1) , pass standard deviation , mean value 
])

trainset = CIFAR10(root ="./data", train=True ,  download = True, transform =transform )
testset = CIFAR10(root ="./data", train=False ,  download = True, transform =transform )

In [3]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [4]:
train_loader = DataLoader(trainset, batch_size = 64, shuffle =True)
test_loader = DataLoader(testset, batch_size = 64)


# Build CNN

In [5]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.convo_layers = nn.Sequential(
            nn.Conv2d(3,32, kernel_size = 3, padding = 1),  # no. of input channels , output channel/ no. of feature map = (n+f-2p)/stride_value +1
            nn.ReLU(),
            nn.MaxPool2d(2,2), # size of kernel , stride

            nn.Conv2d(32,64, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), # size of kernel , stride

            nn.Conv2d(64,128, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2) # size of kernel , stride
        
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256,10)
        )

    def forward(self, x):
        x = self.convo_layers(x)
        x = x.view(x.size(0), -1)  # flatten the inputs
        x = self.fc_layers(x)
        # x = self.Conv2d(x)
        # x = self.ReLU(x)
        # x = self.MaxPool2d(x)
        # x = self.fc_layers(x)
        return x

In [6]:
model = CNN()

In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

# Training of Model

In [ ]:
epochs = 10
train_loss = []
valid_losses = []
for epoch in range(epochs):
    running_loss = 0.0

    for images, labels in train_loader:

        optimizer.zero_grad()

        outputs = model.forward(images)   # No .to(device)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        epoch_train_loss = running_loss / len(train_loader)
        train_loss.append(epoch_train_loss)
        
        # Validation stage
    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():  # no gradient compute
        for xb, yb in test_loader:
            outputs = model(xb) 
            loss = criterion(outputs, yb)
            # during validation no backward propogation is required as we don't have to addjust gradient value
            running_val_loss += loss

        epoch_val_loss =  running_val_loss / len(test_loader)
        valid_losses.append(epoch_val_loss)
        print(f"epoch {epoch+1}/{epochs} ==> train loss = {epoch_train_loss} & val loss = {epoch_val_loss}")


epoch 1/10 ==> train loss = 1.3627952496566431 & val loss = 1.0491269826889038


In [ ]:
# Evaluation of model
correct_labels = 0
total_labels = 0
model.eval()

with torch.no_grad():
    for images , labels in test_loader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)
print(f"accuracy score: {correct_labels / total_labels *100}")